# LLM evaluation
question→hybrid retrieval→Gemini answer→LLM judge

offline A→Q→A′ evaluation pattern: reference answer 𝐴, question 𝑄, and RAG output 𝐴′

In [1]:
from dotenv import load_dotenv
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

os.chdir(PROJECT_ROOT)

print("Working directory:", Path.cwd())
print("Source directory:", PROJECT_ROOT / "src")

load_dotenv(PROJECT_ROOT / ".env")

print("GEMINI_API_KEY configured:", bool(os.getenv("GEMINI_API_KEY")))

Working directory: /Users/camillecu/Downloads/KUL/llm_project/llm_project2026
Source directory: /Users/camillecu/Downloads/KUL/llm_project/llm_project2026/src
GEMINI_API_KEY configured: True


In [4]:

import csv
from pathlib import Path

source = Path("data/retrieval_ground_truth.csv")
target = Path("data/rag_ground_truth.csv")

with source.open(newline="", encoding="utf-8") as src, \
     target.open("w", newline="", encoding="utf-8") as dst:

    reader = csv.DictReader(src)
    writer = csv.DictWriter(
        dst,
        fieldnames=[
            "query",
            "expected_chunk_ids",
            "reference_answer",
        ],
    )
    writer.writeheader()

    for row in reader:
        writer.writerow({
            "query": row["query"],
            "expected_chunk_ids": row["expected_chunk_ids"],
            "reference_answer": "",
        })

print(f"Created {target} with {sum(1 for _ in open(target, encoding='utf-8')) - 1} questions.")


Created data/rag_ground_truth.csv with 30 questions.


In [9]:
import pandas as pd

rag_ground_truth_path = PROJECT_ROOT / "data" / "rag_ground_truth.csv"

df_ground_truth = pd.read_csv(rag_ground_truth_path)

required_columns = {
    "query",
    "expected_chunk_ids",
    "reference_answer",
}


ground_truth = df_ground_truth.to_dict(orient="records")

print("Evaluation questions:", len(ground_truth))

Evaluation questions: 30


In [11]:
print(ground_truth)

[{'query': 'What did Apple say about the upcoming advanced manufacturing center in Houston?', 'expected_chunk_ids': 'AAPL_2026Q3:13', 'reference_answer': 'Location & Current Use: Located in a facility currently used to assemble advanced AI servers.\n\nProduction Expansion: Apple will begin manufacturing the Mac mini there later this year.\n\nPurpose & Goals: The center will teach students, supplier employees, and businesses of all sizes the innovative processes Apple uses to empower American manufacturers and strengthen the U.S. advanced manufacturing ecosystem.'}, {'query': 'What is the Apple Upgrade program and what benefits does it offer to customers?', 'expected_chunk_ids': 'AAPL_2026Q3:30', 'reference_answer': "Program Overview: A leasing plan offered in U.S. retail stores designed to make it easier and more affordable for customers to get Apple's latest devices on a schedule.\n\nBenefits: Leverages the higher residual value of Apple products relative to competitors to provide an 

In [13]:
# Generate RAG answers
from rag_helper_project_rewrite import ask

def source_chunk_ids(sources):
    return [
        f"{source['doc_id']}:{source['chunk_index']}"
        for source in sources
    ]

def generate_rag_answer(record, limit=5):
    result = ask(record["query"], limit=limit)

    return {
        "query": record["query"],
        "expected_chunk_ids": record["expected_chunk_ids"],
        "reference_answer": record["reference_answer"],
        "rag_answer": result["answer"],
        "retrieved_chunk_ids": "|".join(
            source_chunk_ids(result["sources"])
        ),
        "prompt": result["prompt"],
        "limit": limit,
    }

In [14]:
# Test it on one record:
sample = generate_rag_answer(ground_truth[15])

print("Question:", sample["query"])
print("\nReference answer:", sample["reference_answer"])
print("\nRAG answer:", sample["rag_answer"])
print("\nExpected chunks:", sample["expected_chunk_ids"])
print("\nRetrieved chunks:", sample["retrieved_chunk_ids"])


=== RAW QUERY REWRITE RESPONSE ===
'{"search_query":"Meta META Chad Heaton executive participants second quarter 2026 results earnings call","symbols":["META"],"year":2026,"quarter":2}'

=== QUERY REWRITE ===
Original query:   Who joined Chad Heaton to discuss Meta's second quarter 2026 results?
Search query:     Meta META Chad Heaton executive participants second quarter 2026 results earnings call
Symbols:          ['META']
Year:             2026
Quarter:          2
Normalized query for text search: meta or meta or chad or heaton or executive or participants or second or quarter or 2026 or results or earnings or call

=== RETRIEVED CONTEXT ===
[1] META 2026 Q2
Operator : Hello, and welcome to Meta's Second Quarter 2026 Earnings Conference Call. At this time, I would like to welcome everyone to Meta's Second Quarter 2026 Earnings Conference Call. And this call will be recorded. Thank you very much. Chad Heaton, Meta's Vice President of Finance, you may begin.

Chad Heaton : Thank you.

Question: Who joined Chad Heaton to discuss Meta's second quarter 2026 results?

Reference answer: Mark Zuckerberg: Chief Executive Officer (CEO)

Susan Li: Chief Financial Officer (CFO)

RAG answer: Mark Zuckerberg, CEO, and Susan Li, CFO, joined Chad Heaton to discuss Meta's second quarter 2026 results.

Expected chunks: META_2026Q2:0

Retrieved chunks: META_2026Q2:0|META_2026Q2:15|META_2026Q2:40|META_2026Q2:28|META_2026Q2:60


# Generate all answers

In [9]:
import random
import time
from tqdm.auto import tqdm

In [10]:
SLEEP_SECONDS = 32
MAX_RETRIES = 5

def generate_with_retry(record, limit=5):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return generate_rag_answer(record, limit=limit)

        except Exception as exc:
            wait_seconds = min(60, 2 ** attempt) + random.uniform(0, 1)

            print(
                f"\nAttempt {attempt}/{MAX_RETRIES} failed for: "
                f"{record['query'][:60]}..."
            )
            print(f"Error: {exc}")
            print(f"Retrying in {wait_seconds:.1f} seconds...")

            if attempt == MAX_RETRIES:
                return {
                    "query": record["query"],
                    "expected_chunk_ids": record["expected_chunk_ids"],
                    "reference_answer": record["reference_answer"],
                    "rag_answer": "",
                    "retrieved_chunk_ids": "",
                    "prompt": "",
                    "limit": limit,
                    "error": repr(exc),
                }

            time.sleep(wait_seconds)

In [11]:
rag_answers = []

for i, record in enumerate(tqdm(ground_truth), start=1):
    result = generate_with_retry(record, limit=5)
    rag_answers.append(result)

    if i < len(ground_truth):
        time.sleep(SLEEP_SECONDS)

  0%|          | 0/30 [00:00<?, ?it/s]


Attempt 1/5 failed for: What did Apple say about rising memory costs and their impac...
Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 2.7 seconds...

Attempt 1/5 failed for: What did Google say about AI infrastructure demand and suppl...
Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 2.4 seconds...

Attempt 1/5 failed for: How did Alphabet describe the AI opportunity and expected RO...
Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 2.2 seconds...

Attempt 2/5 failed for: How did Alphabet desc

In [12]:
# Save the intermediate output immediately:

df_rag_answers = pd.DataFrame(rag_answers)

answers_path = PROJECT_ROOT / "data" / "rag_answers.csv"
df_rag_answers.to_csv(answers_path, index=False)

print("Saved:", answers_path)
print("Successful answers:", df_rag_answers["rag_answer"].notna().sum())

Saved: /Users/camillecu/Downloads/KUL/llm_project/llm_project2026/data/rag_answers.csv
Successful answers: 30


In [13]:
# import the files for evaluation
import pandas as pd

answers_path = PROJECT_ROOT / "data" / "rag_answers.csv"

df_answers = pd.read_csv(answers_path)

print("Total rows:", len(df_answers))
print("Columns:", df_answers.columns.tolist())

Total rows: 30
Columns: ['query', 'expected_chunk_ids', 'reference_answer', 'rag_answer', 'retrieved_chunk_ids', 'prompt', 'limit', 'error']


In [14]:
# Filter to rows that have both a non-empty RAG response and no error:


successful_mask = (
    df_answers["rag_answer"].notna()
    & df_answers["rag_answer"].astype(str).str.strip().ne("")
)

if "error" in df_answers.columns:
    successful_mask &= (
        df_answers["error"].isna()
        | df_answers["error"].astype(str).str.strip().eq("")
    )

df_successful = df_answers[successful_mask].copy()
df_failed = df_answers[~successful_mask].copy()

print("Successful RAG answers:", len(df_successful))
print("Failed / missing answers:", len(df_failed))

Successful RAG answers: 15
Failed / missing answers: 15


In [15]:
# Save a clean partial-run file
partial_answers_path = PROJECT_ROOT / "data" / "rag_answers_partial.csv"

df_successful.to_csv(partial_answers_path, index=False)

print("Saved:", partial_answers_path)

Saved: /Users/camillecu/Downloads/KUL/llm_project/llm_project2026/data/rag_answers_partial.csv


In [16]:
df_successful[
    [
        "query",
        "reference_answer",
        "rag_answer",
        "expected_chunk_ids",
        "retrieved_chunk_ids",
    ]
].head(5)

,query,reference_answer,rag_answer,expected_chunk_ids,retrieved_chunk_ids
0,What did Apple report for total revenue in fis...,Apple reported fiscal Q3 2026 revenue of $109....,Apple reported $109.4 billion in total revenue...,AAPL_2026Q3:2|AAPL_2026Q3:16,AAPL_2026Q3:21|AAPL_2026Q3:2|AAPL_2026Q3:0|AAP...
1,"What gross margin did Apple report, and what d...",Apple reported total company gross margin of 5...,Apple reported a gross margin of 49.3% for the...,AAPL_2026Q3:16|AAPL_2026Q3:17|AAPL_2026Q3:55|A...,AAPL_2026Q3:56|AAPL_2026Q3:31|AAPL_2026Q3:26|A...
2,What did Apple guide for revenue growth and gr...,Apple guided September-quarter total company r...,Apple guided for total company revenue to grow...,AAPL_2026Q3:25|AAPL_2026Q3:26,AAPL_2026Q3:26|AAPL_2026Q3:2|INTC_2026Q2:12|AA...
3,What supply constraints did Apple expect to af...,Apple expected supply constraints to intensify...,Apple expected supply constraints to affect iP...,AAPL_2026Q3:25|AAPL_2026Q3:29|AAPL_2026Q3:30|A...,AAPL_2026Q3:31|AAPL_2026Q3:30|AAPL_2026Q3:35|A...
4,What did Apple say about rising memory costs a...,Apple said memory costs rose from March to Jun...,"Beyond September, Apple expects to see decreas...",AAPL_2026Q3:32|AAPL_2026Q3:33,AAPL_2026Q3:33|AAPL_2026Q3:32|AAPL_2026Q3:57|A...


In [17]:
from rag_evaluation import RAGAnswerEvaluation, judge_rag_answer

In [18]:
# test the evaluation on one record
test_record = df_successful.iloc[0].to_dict()

test_judgment = judge_rag_answer(test_record)

print(test_judgment.score)
print(test_judgment.failure_type)
print(test_judgment.reasoning)

good
none
The RAG answer directly and accurately provides all the information requested in the question and present in the reference answer, including the total revenue, the year-over-year increase, and the fact that it was a June-quarter record for fiscal Q3 2026.


In [19]:
import random
import time

import pandas as pd
from tqdm.auto import tqdm

JUDGE_OUTPUT_PATH = PROJECT_ROOT / "data" / "rag_evaluations_partial.csv"

JUDGE_SLEEP_SECONDS = 12
MAX_RETRIES = 5


def judge_with_retry(record):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            judgment = judge_rag_answer(record)

            return judgment, None

        except Exception as exc:
            wait_seconds = min(60, 2 ** attempt) + random.uniform(0, 1)

            print(
                f"\nJudge attempt {attempt}/{MAX_RETRIES} failed "
                f"for: {record['query'][:80]}..."
            )
            print(f"Error: {exc}")

            if attempt == MAX_RETRIES:
                return None, repr(exc)

            print(f"Retrying in {wait_seconds:.1f} seconds...")
            time.sleep(wait_seconds)

In [22]:
if JUDGE_OUTPUT_PATH.exists():
    df_existing_judgments = pd.read_csv(JUDGE_OUTPUT_PATH)

    completed_mask = df_existing_judgments["judge_score"].isin(
        ["good", "bad"]
    )

    judged_rows = df_existing_judgments[
        completed_mask
    ].to_dict(orient="records")

    completed_queries = set(
        df_existing_judgments.loc[
            completed_mask,
            "query",
        ]
    )

    print("Saved evaluation rows:", len(df_existing_judgments))
    print("Completed judgments:", len(judged_rows))
else:
    judged_rows = []
    completed_queries = set()

remaining_records = [
    record
    for record in df_successful.to_dict(orient="records")
    if record["query"] not in completed_queries
]

print("Answers still to judge:", len(remaining_records))

Answers still to judge: 15


In [23]:
for i, record in enumerate(
    tqdm(remaining_records),
    start=1,
):
    judgment, error = judge_with_retry(record)

    judged_rows.append({
        **record,
        "judge_score": (
            judgment.score if judgment else "error"
        ),
        "judge_failure_type": (
            judgment.failure_type if judgment else "judge_error"
        ),
        "judge_reasoning": (
            judgment.reasoning if judgment else error
        ),
    })

    pd.DataFrame(judged_rows).to_csv(
        JUDGE_OUTPUT_PATH,
        index=False,
    )

    if i < len(remaining_records):
        time.sleep(JUDGE_SLEEP_SECONDS)

  0%|          | 0/15 [00:00<?, ?it/s]

In [24]:
df_eval = pd.read_csv(JUDGE_OUTPUT_PATH)

scored = df_eval[
    df_eval["judge_score"].isin(["good", "bad"])
].copy()

good_count = (scored["judge_score"] == "good").sum()
bad_count = (scored["judge_score"] == "bad").sum()
good_rate = good_count / len(scored) if len(scored) else 0

print(f"Successful RAG answers available: {len(df_successful)}")
print(f"Judge-scored answers: {len(scored)}")
print(f"Good answers: {good_count}")
print(f"Bad answers: {bad_count}")
print(f"Partial RAG answer quality: {good_rate:.1%}")

Successful RAG answers available: 15
Judge-scored answers: 15
Good answers: 5
Bad answers: 10
Partial RAG answer quality: 33.3%


# after changing the chunk size with rewrite

In [4]:
import random
import time
from tqdm.auto import tqdm

In [15]:
SLEEP_SECONDS = 32
MAX_RETRIES = 5

def generate_with_retry(record, limit=5):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return generate_rag_answer(record, limit=limit)

        except Exception as exc:
            wait_seconds = min(60, 2 ** attempt) + random.uniform(0, 1)

            print(
                f"\nAttempt {attempt}/{MAX_RETRIES} failed for: "
                f"{record['query'][:60]}..."
            )
            print(f"Error: {exc}")
            print(f"Retrying in {wait_seconds:.1f} seconds...")

            if attempt == MAX_RETRIES:
                return {
                    "query": record["query"],
                    "expected_chunk_ids": record["expected_chunk_ids"],
                    "reference_answer": record["reference_answer"],
                    "rag_answer": "",
                    "retrieved_chunk_ids": "",
                    "prompt": "",
                    "limit": limit,
                    "error": repr(exc),
                }

            time.sleep(wait_seconds)

In [ ]:
rag_answers = []

for i, record in enumerate(tqdm(ground_truth[:10]), start=1):
    result = generate_with_retry(record, limit=5)
    rag_answers.append(result)

    if i < len(ground_truth[:10]):
        time.sleep(SLEEP_SECONDS)

In [18]:
# Save the intermediate output immediately:

df_rag_answers = pd.DataFrame(rag_answers)

answers_path = PROJECT_ROOT / "data" / "rag_answers.csv"
df_rag_answers.to_csv(answers_path, index=False)

print("Saved:", answers_path)
print("Successful answers:", df_rag_answers["rag_answer"].notna().sum())

Saved: /Users/camillecu/Downloads/KUL/llm_project/llm_project2026/data/rag_answers.csv
Successful answers: 10
